## Запросы к API

In [17]:
import requests
import pandas as pd

In [18]:
url_visits = "https://data-charts-api.hexlet.app/visits"
url_reg = "https://data-charts-api.hexlet.app/registrations"

params = {
    "begin": "2023-03-01",
    "end": "2023-09-01"
}

In [19]:
response_visits = requests.get(url_visits, params=params)
response_reg = requests.get(url_reg, params=params)

data_visits = response_visits.json()
data_reg = response_reg.json()
#print(data_visits[:2])

df_visits = pd.DataFrame(data_visits)
print(df_visits.head())

print('\n --- \n')

df_reg = pd.DataFrame(data_reg)
print(df_reg.head())

print('\n --- \n')

df_ads = pd.read_csv('./ads.csv') 
print(df_ads.head())

                               visit_id platform  \
0  60db9b85-12b3-447a-b93d-be177181f732      web   
1  9e73a9ba-0c42-4a4d-bf8a-bde5bbe655ed      web   
2  9e73a9ba-0c42-4a4d-bf8a-bde5bbe655ed      web   
3  16dea4d8-2fb7-4c6b-b953-dc2b9686daf5      web   
4  166bc823-c133-41fa-913d-3dbc495b2770      web   

                                          user_agent             datetime  
0  Mozilla/5.0 (Windows NT 10.0; WOW64; Trident/7...  2023-04-05T15:03:40  
1  Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...  2023-04-05T10:59:38  
2  Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...  2023-04-03T14:53:40  
3  Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...  2023-04-05T03:12:36  
4  Mozilla/5.0 (Windows NT 10.0; WOW64; Trident/7...  2023-04-05T11:48:56  

 --- 

              datetime                               user_id  \
0  2023-03-01T07:40:13  2e0f6bb8-b029-4f45-a786-2b53990d37f1   
1  2023-03-01T13:14:00  f007f97c-9d8b-48b5-af08-119bb8f6d9b6   
2  2023-03-01T03:05:50  24ff46a

In [20]:
# --- 2. Обработка визитов: оставляем только последний визит для каждого visit_id ---
# Сначала приводим datetime к типу datetime
df_visits['datetime'] = pd.to_datetime(df_visits['datetime'])

# Сортируем по дате, чтобы последний был в конце
df_visits_sorted = df_visits.sort_values('datetime')

# Оставляем только последний визит на visit_id
df_visits_unique = df_visits_sorted.drop_duplicates(subset='visit_id', keep='last')


# --- 3. Агрегация визитов по дате и платформе ---
# Создаём поле date_group как дату без времени (YYYY-MM-DD)
df_visits_unique['date_group'] = df_visits_unique['datetime'].dt.date

visits_agg = (
    df_visits_unique
    .groupby(['date_group', 'platform'], observed=True)
    .size()
    .reset_index(name='visits')
)


# --- 4. Агрегация регистраций по дате и платформе ---
df_reg['datetime'] = pd.to_datetime(df_reg['datetime'])
df_reg['date_group'] = df_reg['datetime'].dt.date

reg_agg = (
    df_reg
    .groupby(['date_group', 'platform'], observed=True)
    .size()
    .reset_index(name='registrations')
)


# --- 5. Объединение и расчёт конверсии ---
# Объединяем по date_group и platform (LEFT JOIN, чтобы не потерять дни без регистраций)
df_merged = visits_agg.merge(reg_agg, on=['date_group', 'platform'], how='left')

# Заполняем пропуски нулями (если визиты были, а регистраций нет)
df_merged['registrations'] = df_merged['registrations'].fillna(0).astype(int)

# Расчёт конверсии: registrations / visits. Если визитов 0 — конверсия 0
df_merged['conversion'] = (
    df_merged.apply(lambda row: row['registrations'] / row['visits'] if row['visits'] > 0 else 0, axis=1)
)

# Приводим date_group к строке, чтобы JSON был читаемым (дата как строка)
df_merged['date_group'] = df_merged['date_group'].astype(str)


# --- 6. Сохранение в conversion.json ---
df_merged.to_json('./conversion_dict.json', orient='records', force_ascii=False)

print(df_merged.head())

   date_group platform  visits  registrations  conversion
0  2023-03-01  android      75             61    0.813333
1  2023-03-01      ios      22             18    0.818182
2  2023-03-01      web     279              8    0.028674
3  2023-03-02  android      67             59    0.880597
4  2023-03-02      ios      31             24    0.774194


C:\Users\ignatenko_ku\AppData\Local\Temp\ipykernel_45856\2947122943.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_visits_unique['date_group'] = df_visits_unique['datetime'].dt.date


In [21]:
# Обработка визитов: последний визит на visit_id
df_visits['datetime'] = pd.to_datetime(df_visits['datetime'])
df_visits_unique = (
    df_visits
    .sort_values('datetime')
    .drop_duplicates(subset='visit_id', keep='last')
    .copy()
)

# ИСПРАВЛЕНИЕ: группируем по дням, обнуляя время, но сохраняя тип datetime
df_visits_unique['date_group'] = df_visits_unique['datetime'].dt.normalize()

visits_agg = (
    df_visits_unique
    .groupby(['date_group', 'platform'], observed=True)
    .size()
    .reset_index(name='visits')
)

# Агрегация регистраций по дням и платформе
df_reg['datetime'] = pd.to_datetime(df_reg['datetime'])
# ИСПРАВЛЕНИЕ: тоже нормализуем, чтобы дни совпадали
df_reg['date_group'] = df_reg['datetime'].dt.normalize()

reg_agg = (
    df_reg
    .groupby(['date_group', 'platform'], observed=True)
    .size()
    .reset_index(name='registrations')
)

# Объединение и конверсия
df_merged = visits_agg.merge(reg_agg, on=['date_group', 'platform'], how='left')
df_merged['registrations'] = df_merged['registrations'].fillna(0).astype(int)

# Векторизованный расчёт конверсии (быстрее и чище, чем apply)
df_merged['conversion'] = df_merged['registrations'] / df_merged['visits']
df_merged['conversion'] = (df_merged['conversion'] * 100).fillna(0)

# Сохранение в JSON (timestamp-даты, orient=columns)
df_merged.to_json('./conversion.json', orient='columns', force_ascii=False)

print(df_merged[['date_group', 'platform', 'visits', 'registrations', 'conversion']].head(15))

   date_group platform  visits  registrations  conversion
0  2023-03-01  android      75             61   81.333333
1  2023-03-01      ios      22             18   81.818182
2  2023-03-01      web     279              8    2.867384
3  2023-03-02  android      67             59   88.059701
4  2023-03-02      ios      31             24   77.419355
5  2023-03-02      web     515             23    4.466019
6  2023-03-03  android      26             22   84.615385
7  2023-03-03      ios      40             34   85.000000
8  2023-03-03      web     617             51    8.265802
9  2023-03-04  android      94             77   81.914894
10 2023-03-04      bot     184              0    0.000000
11 2023-03-04      ios      68             43   63.235294
12 2023-03-04      web     485             39    8.041237
13 2023-03-05  android      66             54   81.818182
14 2023-03-05      bot     226              0    0.000000


In [23]:

# --- Подготовка рекламы (агрегация по дате) ---
df_ads['date'] = pd.to_datetime(df_ads['date'])
#df_ads['date_group'] = df_ads['date'].dt.date
# Делаем date_group как datetime (полночь дня) вместо date
df_ads['date_group'] = df_ads['date'].dt.normalize()  # это даст datetime64[ns], время = 00:00:00


ads_agg = (
    df_ads
    .groupby('date_group', observed=True)
    .agg(
        cost=('cost', 'sum'),
        utm_campaign=('utm_campaign', lambda x: x.iloc[0] if len(x) > 0 else 'none')
    )
    .reset_index()
)

ads_agg = ads_agg.sort_values('date_group')

# ---  Объединение с конверсиями ---
df_final = df_merged.merge(ads_agg, on='date_group', how='left')

# Заполняем отсутствующие значения (если в день не было рекламы)
df_final['cost'] = df_final['cost'].fillna(0)
df_final['utm_campaign'] = df_final['utm_campaign'].fillna('none')

# Сортировка: сначала по дате, потом по платформе
df_final = df_final.sort_values(['date_group', 'platform'])
print(df_final.head())

# --- Сохранение в JSON ---
df_final.to_json('./ads.json', orient='records', force_ascii=False)


  date_group platform  visits  registrations  conversion   cost  \
0 2023-03-01  android      75             61   81.333333  212.0   
1 2023-03-01      ios      22             18   81.818182  212.0   
2 2023-03-01      web     279              8    2.867384  212.0   
3 2023-03-02  android      67             59   88.059701  252.0   
4 2023-03-02      ios      31             24   77.419355  252.0   

                 utm_campaign  
0  advanced_algorithms_series  
1  advanced_algorithms_series  
2  advanced_algorithms_series  
3  advanced_algorithms_series  
4  advanced_algorithms_series  
